# Bài 1.1

Tái tạo và sửa lỗi NaN loss

Code dưới đây sẽ sinh ra NaN loss. Hãy:

*   Chạy code, quan sát loss trở thành nan tại epoch nào
*   Xác định nguyên nhân (gợi ý: learning rate quá lớn)
*   Sửa để loss hội tụ bình thường

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [4]:
# Simple Model
model = nn.Linear(784,10)
criterion = nn.CrossEntropyLoss()

# With learning rate too high -> loss diverges -> NaN
optimizer = torch.optim.SGD(model.parameters(), lr=100.0)

transform = transforms.Compose([transforms.ToTensor()])
ds = datasets.MNIST(root='./data', download=True, transform=transform)
loader = DataLoader(ds, batch_size=64, shuffle =True)

for epoch in range(5):
    total_loss = 0
    for imgs, labels in loader:
        imgs = imgs.view(imgs.size(0), -1) # Flatten the images
        optimizer.zero_grad() # Clear gradients
        output = model(imgs)
        loss = criterion(output, labels) 
        loss.backward() # to compute gradients
        optimizer.step()
        total_loss += loss.item()
        if torch.isnan(loss):
            print(f"❌ NaN detected! Epoch {epoch+1}")
            break
        
    print(f"Epoch {epoch+1} - Loss: {total_loss/len(loader):4.4f}")
         

Epoch 1 - Loss: 36.2097
Epoch 2 - Loss: 24.8085
Epoch 3 - Loss: 24.9959
Epoch 4 - Loss: 22.8293
Epoch 5 - Loss: 23.5324


Loss quá lớn. Lý do learning rate = 100. model đi quá nhanh, bỏ qua rất nhiều dữu liệu, dẫn đến loss mới cao như vậy

In [5]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

tensor(0.9321)

Gradient quá lơn dẫn đến 2 loại bug:

*   có khả năng đi qua điểm hội tụ
  
*   model không học đủ dữ liệu -> có thể dự đoán sai

In [6]:
assert not torch.isnan(imgs).any(), "Input has NaN values"
assert not torch.isnan(imgs).any(), "Input has Inf values"

In [7]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"end=❌ NaN detected in {name}")

#   FIX

In [10]:
# Simple Model
model = nn.Linear(784,10)
criterion = nn.CrossEntropyLoss()

# With learning rate too high -> loss diverges -> NaN
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

transform = transforms.Compose([transforms.ToTensor()])
ds = datasets.MNIST(root='./data', download=True, transform=transform)
loader = DataLoader(ds, batch_size=128, shuffle =True)

for epoch in range(5):
    total_loss = 0
    for imgs, labels in loader:
        imgs = imgs.view(imgs.size(0), -1) # Flatten the images
        optimizer.zero_grad() # Clear gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        output = model(imgs)
        loss = criterion(output, labels) 
        loss.backward() # to compute gradients
        optimizer.step()
        total_loss += loss.item()
        if torch.isnan(loss):
            print(f"❌ NaN detected! Epoch {epoch+1}")
            break
        
    print(f"Epoch {epoch+1} - Loss: {total_loss/len(loader):4.4f}")
         

Epoch 1 - Loss: 1.2500
Epoch 2 - Loss: 0.7058
Epoch 3 - Loss: 0.5810
Epoch 4 - Loss: 0.5208
Epoch 5 - Loss: 0.4839


*   Sửa learning rate -> đảm bảo model học nhiều dữ liệu hơn -> loss giảm

*   Tăng batch_size -> Tuy sẽ xử lý chậm hơn nhưng học sẽ chính xác hơn

*   Thêm gradient clipping sẽ khiến hàm hội tụ nhanh hơn

# Bài 1.2

Debug Shape Mismatch

Đọc lỗi bên dưới và sửa đúng vị trí:

In [11]:
import torch
import torch.nn as nn

In [12]:
# Model recipe input (B, 512) -> output (B, 10)
model = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256,10)
)

# Wrong shape: input (B, 3, 224, 224) not flattened to (B, 512)
imgs = torch.randn(8, 3, 224, 224) # Simulating a batch of 8 images
# imgs = imgs.view(imgs.size(0), -1) # Flatten the images

try:
    output = model(imgs)
except RuntimeError as e:
    print(f"❌ RuntimeError: {e}")


❌ RuntimeError: mat1 and mat2 shapes cannot be multiplied (5376x224 and 512x256)


FIX

*   flatten img để phù hợp với input của model

In [14]:
# Model recipe input (B, 512) -> output (B, 10)
model = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256,10)
)

imgs = torch.randn(8, 3, 224, 224) # Simulating a batch of 8 images
# Fix wrong shape input
imgs = imgs.view(imgs.size(0), -1) # Flatten the images

try:
    output = model(imgs)
except RuntimeError as e:
    print(f"❌ RuntimeError: {e}")


❌ RuntimeError: mat1 and mat2 shapes cannot be multiplied (8x150528 and 512x256)


In [19]:
# Model recipe input (B, 150528) -> output (B, 10)
# Fix wrong input shape in model definition
model = nn.Sequential(
    nn.Linear(150528, 256),
    nn.ReLU(),
    nn.Linear(256,10)
)

imgs = torch.randn(8, 3, 224, 224) # Simulating a batch of 8 images
# Fix wrong shape input
imgs = imgs.view(imgs.size(0), -1) # Flatten the images

try:
    output = model(imgs)
    print("Output:", output.shape) 
except RuntimeError as e:
    print(f"❌ RuntimeError: {e}")

Output: torch.Size([8, 10])


#   Bài 1.3

Simulate và tránh GPU OOM

Tìm hiểu các kỹ thuật tránh Out of Memory khi train trên GPU:

*   Giảm batch size: thử 256 → 128 → 64

*   Gradient accumulation: train batch nhỏ nhưng update sau N bước

*   Mixed precision: dùng torch.cuda.amp

*   Giải phóng cache: torch.cuda.empty_cache()

In [20]:
import torch
import torch.nn as nn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.cuda.amp as amp
from torch.cuda.amp import GradScaler, autocast

Device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


transform = transforms.Compose([transforms.ToTensor()])
ds = datasets.MNIST(root='./data', download=True, transform=transform)

loader = DataLoader(ds, batch_size=64, shuffle=True)

model = nn.Linear(784, 10)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scaler = GradScaler()
criterion = nn.CrossEntropyLoss()

with autocast("cuda"):
    output = model(imgs)
    loss = criterion(output, labels)

# Gradient accumulation simulates a larger effective batch size
ACCUMULATE_STEPS = 4

optimizer.zero_grad() # Clear gradients before starting training loop

for step, (imgs, labels) in enumerate(loader):
    imgs = imgs.view(imgs.size(0), -1)
    labels = labels

    
    output = model(imgs)
    loss = criterion(output, labels) / ACCUMULATE_STEPS

    scaler.scale(loss).backward()

    if (step + 1) % ACCUMULATE_STEPS == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        if torch.device.type == "cuda":
            torch.cuda.empty_cache()

        print(f"Step {step+1} - Loss: {loss.item():.4f}")


C:\Users\Admin\AppData\Local\Temp\ipykernel_9504\3385976010.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
d:\anaconda\envs\ojt-ai\lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


Step 4 - Loss: 0.5613
Step 8 - Loss: 0.5488
Step 12 - Loss: 0.5467
Step 16 - Loss: 0.5335
Step 20 - Loss: 0.5203
Step 24 - Loss: 0.5170
Step 28 - Loss: 0.4912
Step 32 - Loss: 0.4876
Step 36 - Loss: 0.4879
Step 40 - Loss: 0.4792
Step 44 - Loss: 0.4634
Step 48 - Loss: 0.4738
Step 52 - Loss: 0.4490
Step 56 - Loss: 0.4336
Step 60 - Loss: 0.4199
Step 64 - Loss: 0.4135
Step 68 - Loss: 0.4051
Step 72 - Loss: 0.4167
Step 76 - Loss: 0.4112
Step 80 - Loss: 0.4007
Step 84 - Loss: 0.3703
Step 88 - Loss: 0.3910
Step 92 - Loss: 0.3667
Step 96 - Loss: 0.3513
Step 100 - Loss: 0.3503
Step 104 - Loss: 0.3625
Step 108 - Loss: 0.3489
Step 112 - Loss: 0.3510
Step 116 - Loss: 0.3595
Step 120 - Loss: 0.3209
Step 124 - Loss: 0.3021
Step 128 - Loss: 0.3223
Step 132 - Loss: 0.3219
Step 136 - Loss: 0.3070
Step 140 - Loss: 0.3091
Step 144 - Loss: 0.3120
Step 148 - Loss: 0.3024
Step 152 - Loss: 0.2971
Step 156 - Loss: 0.2831
Step 160 - Loss: 0.2993
Step 164 - Loss: 0.3082
Step 168 - Loss: 0.2827
Step 172 - Loss: 0

* Giảm batch size giúp tránh OOM khi GPU bộ nhớ nhỏ.
* Gradient accumulation cho phép dùng batch hiệu quả lớn hơn bằng cách cập nhật sau N bước.
* Mixed precision "torch.cuda.amp" giảm bộ nhớ và tăng tốc trên GPU hỗ trợ.
* "torch.cuda.empty_cache()" giải phóng bộ nhớ không còn sử dụng giữa các bước.
* Autocast sẽ đảm nhiệm phép toán nào dùng float16 / float32. còn scaler sẽ giải quyết vấn đề gradient quá nhỏ (underflow) khi tính toán với float16 bằng cách scale loss trước khi backward.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Setup data
transform = transforms.Compose([transforms.ToTensor()])
ds = datasets.MNIST(root='./data', download=True, transform=transform)
loader = DataLoader(ds, batch_size=256, shuffle=True)

# Model
model = nn.Linear(784, 10).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# GradScaler: Tự động điều chỉnh loss scaling factor
scaler = GradScaler(enabled=(device.type == "cuda"))

ACCUMULATE_STEPS = 4   # simulate batch_size * 4
optimizer.zero_grad() # Clear gradients before starting training loop

for epoch in range(2):
    total_loss = 0
    for step, (imgs, labels) in enumerate(loader):
        imgs = imgs.view(imgs.size(0), -1)
        output = model(imgs)
        loss = criterion(output, labels) / ACCUMULATE_STEPS # scale loss

        # Forward pass: float16 computation (nếu GPU support)
        with autocast(enabled=(device.type == "cuda")):
            output = model(imgs)
            loss = criterion(output, labels)

        # Backward pass: float16 gradient
        scaler.scale(loss).backward()

        # Optimizer step: float32 parameter update
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if (step + 1) % ACCUMULATE_STEPS == 0:
            print(f"Epoch {epoch+1}, Step {step+1} - Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} - Avg Loss: {total_loss / len(loader):.4f}\n")


Training on: cpu


C:\Users\Admin\AppData\Local\Temp\ipykernel_16336\2583113060.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
C:\Users\Admin\AppData\Local\Temp\ipykernel_16336\2583113060.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):


Epoch 1, Step 4 - Loss: 2.2120
Epoch 1, Step 8 - Loss: 2.0399
Epoch 1, Step 12 - Loss: 1.8663
Epoch 1, Step 16 - Loss: 1.7331
Epoch 1, Step 20 - Loss: 1.5740
Epoch 1, Step 24 - Loss: 1.4882
Epoch 1, Step 28 - Loss: 1.4140
Epoch 1, Step 32 - Loss: 1.2697
Epoch 1, Step 36 - Loss: 1.1730
Epoch 1, Step 40 - Loss: 1.0765
Epoch 1, Step 44 - Loss: 0.9406
Epoch 1, Step 48 - Loss: 0.9518
Epoch 1, Step 52 - Loss: 0.9441
Epoch 1, Step 56 - Loss: 0.7548
Epoch 1, Step 60 - Loss: 0.7410
Epoch 1, Step 64 - Loss: 0.7623
Epoch 1, Step 68 - Loss: 0.7538
Epoch 1, Step 72 - Loss: 0.7448
Epoch 1, Step 76 - Loss: 0.6102
Epoch 1, Step 80 - Loss: 0.6678
Epoch 1, Step 84 - Loss: 0.6366
Epoch 1, Step 88 - Loss: 0.6337
Epoch 1, Step 92 - Loss: 0.7025
Epoch 1, Step 96 - Loss: 0.5663
Epoch 1, Step 100 - Loss: 0.5923
Epoch 1, Step 104 - Loss: 0.5613
Epoch 1, Step 108 - Loss: 0.4895
Epoch 1, Step 112 - Loss: 0.4383
Epoch 1, Step 116 - Loss: 0.5107
Epoch 1, Step 120 - Loss: 0.5292
Epoch 1, Step 124 - Loss: 0.5937
Epo